# Feature Selection & Dimension Reduction - Exercise

In this exercise you will work with the **handwritten digits** dataset that is built into scikit-learn (`load_digits`). Every observation is a small grey-scale image of **8 x 8 pixels** showing a handwritten digit (0-9). Each pixel is one feature with a value between 0 (white) and 16 (black), so every image is described by **64 features** (`pixel_0_0` ... `pixel_7_7`).

The central question of this case: **do we really need all 64 pixels to recognise a digit?**

You will tackle this question in two different ways, just like in class:
1. **Feature Selection** with **Boruta**: keep only the original pixels that are truly relevant.
2. **Dimension Reduction** with **PCA**: replace the 64 pixels by a smaller number of new features (principal components).

For both approaches you will compare the model performance with the performance on all features.

---
---
---
## Complete the tasks in bold

## 0 - Setup

**TASK: Run the cell below to import the libraries you will need.**

In [ ]:
import time  # To time model training
import numpy as np  # To do some calculations
import pandas as pd  # To work with dataframes
import matplotlib.pyplot as plt  # To make plots

from sklearn.datasets import load_digits  # The dataset (built into scikit-learn)
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Install missing package in the notebook environment if needed
#%pip install boruta
from boruta import BorutaPy

RANDOM_STATE = 42  # Use this everywhere for reproducible results

## 1 - Data Loading and Exploration

**TASK: Load the digits dataset with `load_digits(return_X_y=True, as_frame=True)` into `X` and `y`. Also store the full dataset object `load_digits()` in a variable called `digits` (it contains the images in `digits.images`). Print the shape of `X` and show the first rows. *(Expected: 1797 rows and 64 columns.)***

**TASK: Check the class distribution of the target `y`. Is the dataset balanced?**

**TASK: Plot the first 10 images (`digits.images`) in a 2 x 5 grid with `plt.imshow(..., cmap='gray_r')` and use the true label as the title of each image.**

**TASK: Calculate the standard deviation of every pixel (`X.std()`), reshape it to 8 x 8 and show it as an image with a colour bar. Also print the names of the pixels that are constant (standard deviation = 0). *(In the solution we found 3 constant pixels.)***

**QUESTION: Which pixels vary the least? Why does that make sense for images of handwritten digits? Which pixels do you expect a feature selection method to remove?**

*Your answer:*

## 2 - Data Processing

Feature selection (and scaling / PCA) are part of the model: they must be **learned on the training data only**. If we let Boruta or PCA look at the test set, information from the test set leaks into the model and our performance estimate becomes too optimistic (more about this in week 6 - Data Leakage). So we split the data first.

**TASK: Split the data into a training set (75%) and a test set (25%) with `train_test_split`. Use `stratify=y` and `random_state=RANDOM_STATE`. Print the shapes. *(Expected: 1347 training and 450 test observations.)***

## 3 - Feature Selection with Boruta

Recall how **Boruta** works:
- For every feature a **shadow feature** is created by randomly shuffling its values (so it has no relation with the target any more).
- A random forest is trained on the real + shadow features, and the importance of each real feature is compared with the **best shadow feature**.
- This is repeated many times. Features that are significantly more important than random noise are **confirmed** (rank 1), features that are clearly worse are **rejected**, and undecided ones are **tentative** (rank 2).

**TASK: Define a `RandomForestClassifier` (`n_jobs=-1`, `class_weight='balanced'`, `max_depth=5`, `random_state=RANDOM_STATE`) and wrap it in a `BorutaPy` feature selector (`n_estimators='auto'`, `max_iter=50`, `random_state=RANDOM_STATE`). Fit the selector on the training set only. Remember to use `.values` to pass numpy arrays. *(This takes about 15-30 seconds.)***

**TASK: Print the number of selected (confirmed) features and the number of tentative features. *(In the solution Boruta kept 50 of the 64 pixels and found 0 tentative features.)***

**TASK: Build a dataframe `feature_ranks` with the columns `feature`, `rank` (`ranking_`) and `keep` (`support_`), sorted by rank. Show the rejected features.**

**TASK: Visualise the result: show the average digit of the training set (`X_train.mean()` reshaped to 8 x 8) next to the Boruta mask (`feat_selector.support_` reshaped to 8 x 8, e.g. with `cmap='RdYlGn'`).**

**QUESTION: Which pixels did Boruta reject? Does this match your expectation from the standard deviation plot in section 1? Why are these pixels irrelevant?**

*Your answer:*

## 4 - Model Performance: All Features vs Selected Features

**TASK: Create `X_train_sel` and `X_test_sel` that contain only the features selected by Boruta. Then write a helper function `evaluate_model(name, model, X_tr, X_te, y_tr, y_te)` that returns a dictionary with the model name, the number of features, the 5-fold cross-validated accuracy on the training set (`cross_val_score`), the test accuracy and the time needed to fit the model. Create an empty list `results` to store all results.**

**TASK: Evaluate a `RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)` on all features and on the Boruta features, append both results to `results` and show them as a dataframe. *(In the solution we got a test accuracy ≈ 0.967 with all features and ≈ 0.971 with the selected features.)***

**QUESTION: Compare the two models.**
- Did removing the rejected pixels hurt the performance?
- What are the advantages of a model with fewer features, even if the accuracy stays the same?
- On which kind of datasets would you expect feature selection to make a bigger difference?

*(Hint: small differences in accuracy (e.g. 1-2 images on 450) are not meaningful.)*

*Your answer:*

## 5 - Dimension Reduction with PCA

PCA looks for the directions with the **largest variance**. Features with a larger scale would dominate these directions, so (just like in class) we **standardize** the data first. The scaler is fitted on the training set only and then applied to the test set.

**TASK: Standardize the data with a `StandardScaler`: fit it on `X_train` and transform both `X_train` and `X_test` (store them in `X_train_scaled` and `X_test_scaled`). Check that the scaled training features have mean 0 and standard deviation 1.**

**TASK: Fit a `PCA` with all components on `X_train_scaled`. Store the `explained_variance_ratio_` and its cumulative sum (`np.cumsum`) in `explained_variance_ratio` and `cumulative_variance`. Print the explained variance of the first 5 components and the variance explained by the first two components together. *(In the solution PC1 explains ≈ 0.12 and PC1 + PC2 ≈ 0.22 of the variance.)***

**TASK: Make a figure with two subplots: the Cumulative Variance Plot (with horizontal lines at 90% and 95%) and the Scree Plot (explained variance ratio per component).**

**TASK: Determine how many components are needed to explain 80%, 90%, 95% and 99% of the variance. Store the number of components for 95% in `n_components_95`. *(In the solution: 21, 31, 40 and 54 components.)***

**QUESTION: Interpret the scree plot and the cumulative variance plot. Is there a clear elbow? How many components would you keep and how much of a reduction is that compared to the 64 original pixels?**

*Your answer:*

**TASK: Transform `X_train_scaled` with the fitted PCA and make a scatter plot of the first two principal components, coloured by the digit (`c=y_train`, `cmap='tab10'`). Add axis labels, a title and a colour bar.**

**TASK: Each principal component is a linear combination of the 64 pixels. Visualise the loadings (`pca.components_`) of the first three components as 8 x 8 images (use a diverging colour map such as `'coolwarm'`).**

**QUESTION: Interpret the 2D scatter plot and the loadings.**
- Which digits are well separated with only two components, and which ones overlap?
- PC1 and PC2 together explain only about 22% of the variance. Do you expect that a model on 2 components can classify the digits well?
- Can you still say which original pixels are important after PCA? Compare with Boruta.

*Your answer:*

**TASK: Effect of scaling: fit a PCA on the unscaled `X_train` and calculate how many components are needed for 95% of the variance. Compare with the scaled version. *(In the solution: 29 components without scaling vs 40 with scaling.)***

**QUESTION: All pixels are measured on the same scale (0-16). Why do you need a different number of components with and without scaling? Is standardization always necessary before PCA? When is it essential?**

*Your answer:*

## 6 - Model Performance on Principal Components

We now train models on the principal components. By putting the scaler, the PCA and the model in a **pipeline**, scaling and PCA are re-fitted inside every cross-validation fold, so no information leaks from the validation folds. We use a **logistic regression**, a linear model that often benefits from PCA, and also a random forest.

**TASK: Build three pipelines with `make_pipeline` and evaluate them with `evaluate_model` (on the unscaled `X_train` / `X_test`, the pipeline does the scaling): (1) `StandardScaler` + `LogisticRegression(max_iter=5000)` on all pixels, (2) `StandardScaler` + `PCA(n_components=n_components_95)` + `LogisticRegression`, (3) `StandardScaler` + `PCA(n_components=n_components_95)` + `RandomForestClassifier(n_estimators=200)`. Set `n_features` to the number of components for the PCA models and append all results. *(In the solution we got test accuracies ≈ 0.978, ≈ 0.960 and ≈ 0.951.)***

**TASK: Investigate how the accuracy depends on the number of components: loop over `[2, 5, 10, 15, 20, 30, 40, 50, 64]`, fit the `StandardScaler` + `PCA(n_components=k)` + `LogisticRegression` pipeline and store the test accuracy. Print the results and plot the accuracy against the number of components. *(In the solution: ≈ 0.51 with 2 PCs and ≈ 0.93 with 15 PCs.)***

**QUESTION: Interpret the results of the PCA models.**
- How does the accuracy evolve with the number of components? Where does adding components stop paying off?
- Did PCA improve the performance compared to all features? Why (not)?
- Why is the random forest on PCs worse than the random forest on the original (Boruta) pixels?

*Your answer:*

## 7 - Comparison of all approaches

**TASK: Combine all results in one dataframe `results_df`, sorted by test accuracy.**

**QUESTION: Which approach would you choose in each of the following situations? Motivate your answer.**
- (a) A doctor / business user wants to know **which original variables** drive the prediction.
- (b) You need a **compact, fast** model and interpretability is not important.
- (c) You want to **visualise** a high-dimensional dataset in 2D.

*Your answer:*

### Summary

**QUESTION: Summarise what you learned in this exercise. Consider the difference between feature selection and dimension reduction, the effect on model performance and the role of scaling.**

*Your answer:*

## Great work!

----